# InsureAI — Module 4: RNN / LSTM (Customer Review Sentiment Analysis)
### Capstone Project: End-to-End AI Assistant for Insurance

---
### Business Problem & Objectives:
- Automatically assess customer satisfaction and emotional tone from policy and claim feedback.
- Classify feedback into Positive vs Negative sentiment using an LSTM architecture.
- Target: Test Accuracy $\ge 85\%$.
- Verify with real-world test sentences (e.g., 'My claim was settled very fast' vs 'Waited 3 months with no response').


In [ ]:
# 1. Setup & Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Bidirectional, Dense, Dropout, Input
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

print('TensorFlow Version:', tf.__version__)


In [ ]:
# Load Reviews Dataset
df_rev = pd.read_csv('../data/insurance_reviews.csv')
print(f'Reviews dataset shape: {df_rev.shape}')
display(df_rev.head())


In [ ]:
# Tokenization & Preprocessing Pipeline
X_train, X_test, y_train, y_test = train_test_split(
    df_rev['review_text'].values, df_rev['sentiment'].values, test_size=0.2, random_state=42, stratify=df_rev['sentiment'].values
)

vocab_size = 5000
max_len = 80
tokenizer = Tokenizer(num_words=vocab_size, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train)

# Pre-padding is critical for LSTM memory cell retention
X_train_pad = pad_sequences(tokenizer.texts_to_sequences(X_train), maxlen=max_len, padding='pre')
X_test_pad = pad_sequences(tokenizer.texts_to_sequences(X_test), maxlen=max_len, padding='pre')
print(f'Padded training shape: {X_train_pad.shape}')


In [ ]:
# Build LSTM Architecture
lstm_model = Sequential([
    Input(shape=(max_len,)),
    Embedding(vocab_size, 64),
    Bidirectional(LSTM(32)),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

lstm_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
lstm_model.summary()

history_lstm = lstm_model.fit(
    X_train_pad, y_train,
    epochs=5,
    batch_size=32,
    validation_split=0.1,
    verbose=1
)


In [ ]:
# Model Evaluation & Test Sentences
y_prob = lstm_model.predict(X_test_pad).flatten()
y_pred = (y_prob >= 0.5).astype(int)
print('Test Accuracy:', round(accuracy_score(y_test, y_pred), 4))
print('Test F1-Score:', round(f1_score(y_test, y_pred), 4))

test_samples = [
    'My claim was settled very fast and the support team was genuinely helpful throughout.',
    'Waited 3 months with no response. Worst customer service ever encountered.',
    'Service was okay but the claim process could be much simpler. Not bad, not great.'
]
seqs = pad_sequences(tokenizer.texts_to_sequences(test_samples), maxlen=max_len, padding='pre')
preds = lstm_model.predict(seqs).flatten()
for s, p in zip(test_samples, preds):
    lbl = 'Positive' if p >= 0.5 else 'Negative'
    print(f'Sentence: "{s}" -> {lbl} ({p*100:.1f}%)')
